```mermaid
flowchart LR
    A0["00"] --> A1a["01a"] --> A1b["01b"] --> A2["02"] --> A3["03"] --> A4a["04a"] --> A4b["04b"]
    A4b --> A5a["05a"] --> A5b["05b"] --> A6a["06a"] --> A6b["06b"]
    A6b --> A7["07"] --> A8a["08a"] --> A8b["08b"]
    A8b --> A9["09"] --> A10["10"] --> A11["11"] --> A12["12"] 
    
    classDef normal fill:#f8f9fa,stroke:#adb5bd,stroke-width:1px,color:#111;
    classDef done fill:#e8f7f0,stroke:#198754,stroke-width:1.5px,color:#111;
    classDef current fill:#fff3cd,stroke:#ff8c00,stroke-width:2px,color:#111;
    
    class A0,A1a,A1b,A2,A3,A4a,A4b,A5a done;
    class A5b current;
    class A6a,A6b,A7,A8a,A8b,A9,A10,A11,A12 normal;
```

# Notebook 05b — Conceptual relation extraction and analysis
 
**Purpose:**  
This notebook loads the annotated corpus (produced by Notebook 05a) and extracts interpretable linguistic patterns:

 - Part‑of‑speech distribution  
 - Adjective–noun relations (e.g., *moral law*)  
 - Subject–verb–object triples (e.g., *man seeks truth*)  
 - Noun compounds (e.g., *state power*)  
 - Concept co‑occurrence windows (e.g., *god – man*)  
 - **Temporal evolution** of the top relations over time  
 
**Prerequisites:**  
 - You have run Notebook 05a, which created the `data/processed/nb05-corpus-split/` folder.  
 - You have the `en_core_web_sm` spaCy model installed (used only for its vocabulary).  
 
**Note:** This notebook runs quickly because it only iterates over pre‑annotated spacy `Doc` objects.

In [ ]:
# -----------------------------
# Import
# -----------------------------
from __future__ import annotations

from pathlib import Path
from collections import Counter

import pandas as pd
import numpy as np
import spacy
from spacy.tokens import DocBin
from tqdm.auto import tqdm

import matplotlib.pyplot as plt
import seaborn as sns

# -----------------------------
# Paths
# -----------------------------
PROJECT_ROOT = Path('.')

DATA_DIR = PROJECT_ROOT / 'data'
PROCESSED_DIR = DATA_DIR / 'processed'

ANALYSIS_DIR = PROJECT_ROOT / 'analysis'
TABLES_DIR = ANALYSIS_DIR / 'tables'
FIGURES_DIR = ANALYSIS_DIR / 'figures'

# Split .spacy files from Notebook 05a
SPLIT_DIR = PROCESSED_DIR / 'nb05-corpus-split'

# Metadata tables from Notebook 05a
META_PATH = TABLES_DIR / 'nb05-doc_annotation_summary.csv'
AGG_META_PATH = TABLES_DIR / 'nb05-doc_agg_annotation_summary.csv'

print('SPLIT_DIR:', SPLIT_DIR)
print('AGG_META_PATH:', AGG_META_PATH)

# -----------------------------
# Plotting parameters
# -----------------------------
sns.set_style("whitegrid")
plt.rcParams["figure.figsize"] = (12, 6)

In [ ]:
# Helper to extract start year from time_bin string and introduce a chronological order
def get_start_year(b: str) -> int:
    try:
        return int(b.split('–')[0] if '–' in b else b.split('-')[0])
    except (ValueError, IndexError, AttributeError):
        return 0

## Load the spaCy Vocabulary
 
We need a spaCy model with a vocabulary that knows POS tags and dependency labels.
We load the same model that was used for annotation; we don't need the full pipeline,
but we need the vocab and the tag/parser strings.

In [ ]:
nlp = spacy.load("en_core_web_sm")
print("\nLoaded model with vocab. Pipeline components:", nlp.pipe_names)

files = list(SPLIT_DIR.glob("*.spacy"))
print(f"\nFound {len(files)} split files in {SPLIT_DIR}.")
if not files:
    raise FileNotFoundError("No .spacy files found. Did you run Notebook 05a?")

## 1. Part‑of‑Speech (POS) distribution

Count all POS tags across the entire corpus. This gives a high‑level stylistic profile.

In [ ]:
pos_counter = Counter()

for filepath in tqdm(files, desc="Loading POS"):
    docbin = DocBin().from_disk(filepath)
    for doc in docbin.get_docs(nlp.vocab):
        for token in doc:
            if not token.is_space:
                pos_counter[token.pos_] += 1

pos_df = pd.DataFrame(pos_counter.items(), columns=['pos', 'count'])
pos_df = pos_df.sort_values('count', ascending=False).reset_index(drop=True)

print("\nTop 15 POS tags:")
display(pos_df.head(15))
pos_df.to_csv(TABLES_DIR / 'nb05-pos_distribution.csv', index=False)
print("Saved POS distribution to", TABLES_DIR / 'nb05-pos_distribution.csv')

## 2. Adjective modifyers (amod)

Extract dependency relations where an adjective modifies a noun (amod).
Examples: "moral law", "human reason".

In [ ]:
adj_noun_rows = []

for filepath in tqdm(files, desc="Extracting amod relations"):
    docbin = DocBin().from_disk(filepath)
    for doc in docbin.get_docs(nlp.vocab):
        pg_id = doc.user_data.get('pg_id', pd.NA)
        title = doc.user_data.get('title', pd.NA)
        time_bin = doc.user_data.get('time_bin', pd.NA)
        
        for token in doc:
            if token.dep_ == 'amod' and token.head.pos_ == 'NOUN':
                adj = token.lemma_.lower()
                noun = token.head.lemma_.lower()
                adj_noun_rows.append({
                    'pg_id': pg_id,
                    'title': title,
                    'time_bin': time_bin,
                    'adjective': adj,
                    'noun': noun,
                    'relation': f'{adj} -> {noun}'
                })

adj_noun_df = pd.DataFrame(adj_noun_rows)
print(f"\nExtracted {len(adj_noun_df)} adjective–noun relations.")

adj_counts = adj_noun_df['relation'].value_counts().reset_index()
adj_counts.columns = ['relation', 'count']
display(adj_counts.head(20))

adj_counts.to_csv(TABLES_DIR / 'nb05-adj_noun_relations.csv', index=False)
print("\nSaved adjective–noun relations to", TABLES_DIR / 'nb05-adj_noun_relations.csv')

## 3. Subject–Verb–Object (SVO) triples
 
Extract triples where a verb has a nominal subject and a direct object.
Examples: "man seeks truth", "reason guides action".

In [ ]:
svo_rows = []

for filepath in tqdm(files, desc="Extracting SVO triples"):
    docbin = DocBin().from_disk(filepath)
    for doc in docbin.get_docs(nlp.vocab):
        pg_id = doc.user_data.get('pg_id', pd.NA)
        title = doc.user_data.get('title', pd.NA)
        time_bin = doc.user_data.get('time_bin', pd.NA)
        
        for token in doc:
            if token.pos_ != 'VERB':
                continue
            
            subj = None
            obj = None
            
            for child in token.children:
                if child.dep_ in ('nsubj', 'nsubjpass'):
                    subj = child.lemma_.lower()
                elif child.dep_ in ('dobj', 'obj'):
                    obj = child.lemma_.lower()
            
            if subj and obj:
                svo_rows.append({
                    'pg_id': pg_id,
                    'title': title,
                    'time_bin': time_bin,
                    'subject': subj,
                    'verb': token.lemma_.lower(),
                    'object': obj,
                    'triple': f'{subj} -> {token.lemma_.lower()} -> {obj}'
                })

svo_df = pd.DataFrame(svo_rows)
print(f"\nExtracted {len(svo_df)} SVO triples.")

svo_counts = svo_df['triple'].value_counts().reset_index()
svo_counts.columns = ['triple', 'count']
display(svo_counts.head(20))

svo_counts.to_csv(TABLES_DIR / 'nb05-svo_triples.csv', index=False)
print("\nSaved SVO triples to", TABLES_DIR / 'nb05-svo_triples.csv')

## 4. Compound nouns

Extract noun–noun compounds, e.g., "state power", "knowledge system".
These often represent domain‑specific concepts.

In [ ]:
compound_rows = []

for filepath in tqdm(files, desc="Extracting compound nouns"):
    docbin = DocBin().from_disk(filepath)
    for doc in docbin.get_docs(nlp.vocab):
        pg_id = doc.user_data.get('pg_id', pd.NA)
        title = doc.user_data.get('title', pd.NA)
        time_bin = doc.user_data.get('time_bin', pd.NA)
        
        for token in doc:
            if token.dep_ == 'compound' and token.head.pos_ == 'NOUN':
                comp = token.lemma_.lower()
                head = token.head.lemma_.lower()
                compound_rows.append({
                    'pg_id': pg_id,
                    'title': title,
                    'time_bin': time_bin,
                    'compound': comp,
                    'head': head,
                    'relation': f'{comp} {head}'
                })

compound_df = pd.DataFrame(compound_rows)
print(f"Extracted {len(compound_df)} noun compounds.")

compound_counts = compound_df['relation'].value_counts().reset_index()
compound_counts.columns = ['relation', 'count']
display(compound_counts.head(20))

compound_counts.to_csv(TABLES_DIR / 'nb05-noun_compounds.csv', index=False)
print("Saved noun compounds to", TABLES_DIR / 'nb05-noun_compounds.csv')

## 5. Concept co‑occurrence windows
 
Slide a fixed‑size window over each document and count unordered pairs of lemmas
that appear together. This captures local conceptual associations.

In [ ]:
WINDOW_SIZE = 5   # number of words to look ahead from the current word

cooc_counter = Counter()

for filepath in tqdm(files, desc="Computing co‑occurrence"):
    docbin = DocBin().from_disk(filepath)
    for doc in docbin.get_docs(nlp.vocab):
        lemmas = [t.lemma_.lower() for t in doc if t.is_alpha and not t.is_stop]
        n = len(lemmas)
        
        for i in range(n - 1):
            a = lemmas[i]
            end = min(i + WINDOW_SIZE + 1, n)
            for j in range(i + 1, end):
                b = lemmas[j]
                if a == b:
                    continue
                if a < b:
                    pair = (a, b)
                else:
                    pair = (b, a)
                cooc_counter[pair] += 1

print(f"\nFound {len(cooc_counter)} unique co‑occurring pairs.")

top_pairs = cooc_counter.most_common(50)
cooc_df = pd.DataFrame([(a, b, c) for (a, b), c in top_pairs],
                       columns=['term_a', 'term_b', 'count'])
display(cooc_df.head(20))

cooc_df.to_csv(TABLES_DIR / 'nb05-cooccurrence_pairs.csv', index=False)
print("\nSaved co‑occurrence pairs to", TABLES_DIR / 'nb05-cooccurrence_pairs.csv')

## 6. Temporal Evolution of Conceptual Relations
 
We now explore how the extracted relations change across historical time bins.
This is crucial for understanding conceptual shifts in the corpus.
 
We normalize raw counts by the total number of tokens in each time period,
because the number of documents per period varies greatly.

In [ ]:
# -----------------------------------------------------------------------------
# 6a. Load the aggregated metadata to get total tokens per time bin
# -----------------------------------------------------------------------------
agg_meta = pd.read_csv(AGG_META_PATH)
agg_meta = agg_meta.dropna(subset=['time_bin'])   # remove docs without date

tokens_per_time = agg_meta.groupby('time_bin')['n_tokens'].sum()

# Sort chronologically by start year
tokens_per_time = tokens_per_time.sort_index(key=lambda idx: idx.map(get_start_year))

print("Tokens per time period (chronological):")
print(tokens_per_time)

In [ ]:
# -----------------------------------------------------------------------------
# 6b. Helper function to compute and plot temporal trends
# -----------------------------------------------------------------------------

def plot_temporal_trends(
    df,
    relation_col,
    top_n=10,
    title="Temporal Trends",
    save_name=None,
    filter_term=None,
    show_pivot=True, 
):
    """
    Plot temporal trends for relations, optionally filtering by a term.

    Parameters
    ----------
    df : pd.DataFrame
        Must contain `relation_col` and `time_bin` columns.
    relation_col : str
        Name of the column with the relation strings (e.g., 'relation', 'triple').
    top_n : int, default 10
        Number of top relations to plot.
    title : str, default "Temporal Trends"
        Plot title.
    save_name : str, optional
        Filename to save the figure.
    filter_term : str, optional
        If provided, only keep rows where `relation_col` contains this term
        (case‑insensitive) before computing top trends.
    show_pivot : bool, default True
        If True, display the pivot table (DataFrame) in the notebook output.
        Set to False to suppress the table (e.g., if you want to display it yourself).
    """
    df_clean = df.dropna(subset=['time_bin']).copy()
    if df_clean.empty:
        print("No data available for plotting after dropping missing time_bin.")
        return

    # ---- Optional: filter by term ----
    if filter_term is not None:
        filter_term = filter_term.lower()
        mask = df_clean[relation_col].str.lower().str.contains(filter_term, na=False)
        df_clean = df_clean[mask]
        if df_clean.empty:
            print(f"No relations containing '{filter_term}' found.")
            return
        title = f"{title} (containing '{filter_term}')"

    # Count occurrences per time_bin and relation
    counts = df_clean.groupby(['time_bin', relation_col]).size().reset_index(name='count')
    counts = counts.merge(tokens_per_time, left_on='time_bin', right_index=True)

    # Relative frequency: per 1,000,000 tokens
    counts['rel_freq'] = (counts['count'] / counts['n_tokens']) * 1_000_000

    # Select top N relations overall (among the filtered set)
    top_relations = (
        counts.groupby(relation_col)['count']
        .sum()
        .sort_values(ascending=False)
        .head(top_n)
        .index
    )

    plot_data = counts[counts[relation_col].isin(top_relations)].copy()
    plot_data['start_year'] = plot_data['time_bin'].str.split('–').str[0].astype(float)
    plot_data = plot_data.sort_values('start_year')

    # Pivot for plotting
    pivot = plot_data.pivot(index='time_bin', columns=relation_col, values='rel_freq')

    # Plot
    ax = pivot.plot(marker='o', linestyle='-', linewidth=2, markersize=6, figsize=(14, 7))
    ax.set_title(title, fontsize=16)
    ax.set_xlabel("Time Period", fontsize=12)
    ax.set_ylabel("Frequency (per 1,000,000 words)", fontsize=12)
    ax.legend(title=relation_col, bbox_to_anchor=(1.05, 1), loc='upper left')
    plt.xticks(rotation=45)
    plt.tight_layout()

    if save_name:
        plt.savefig(FIGURES_DIR / save_name, dpi=300, bbox_inches='tight')
        print(f"Figure saved to {FIGURES_DIR / save_name}")

    # Display pivot if requested
    if show_pivot:
        display(pivot.round(2))
    
    return pivot

In [ ]:
# -----------------------------------------------------------------------------
# 6c. Apply to each relation type
# -----------------------------------------------------------------------------

# Adjective → Noun
if len(adj_noun_df) > 0:
    adj_pivot = plot_temporal_trends(
        adj_noun_df,
        relation_col='relation',
        top_n=8,
        title="Top Adjective→Noun relations over time",
        save_name='nb05-adj_noun_temporal.png'
    )

# SVO Triples
if len(svo_df) > 0:
    svo_pivot = plot_temporal_trends(
        svo_df,
        relation_col='triple',
        top_n=6,
        title="Top Subject→Verb→Object triples over time",
        save_name='nb05-svo_temporal.png'
    )

# Noun Compounds
if len(compound_df) > 0:
    compound_pivot = plot_temporal_trends(
        compound_df,
        relation_col='relation',
        top_n=8,
        title="Top noun compounds over time",
        save_name='nb05-compounds_temporal.png'
    )

### Analysis around a specific term

In [ ]:
# For adjective‑noun relations containing "nature"
plot_temporal_trends(
    adj_noun_df,             
    relation_col='relation',
    top_n=8,
    title="Top adj→noun relations with 'nature'",
    save_name="nb05-nature_relations.png",
    filter_term="nature"
)

# For SVO triples containing "nature"
plot_temporal_trends(
    svo_df,
    relation_col='triple',
    top_n=6,
    filter_term="nature"
)

# Without filter (original behaviour)
plot_temporal_trends(
    compound_df,
    relation_col='relation',
    top_n=10,
    title="Top noun compounds with nature over time",
    filter_term="nature"
)

## Interpreting the Temporal Plots
 
 - **Normalization**: We divide by the total tokens in each period, giving a *relative frequency* (per 1 million words) for fair comparison.
 - **Rising curves**: A concept becomes more prominent over time.
 - **Falling curves**: A concept fades.
 - **U‑shaped or inverted‑U**: A concept peaks in a specific period.
 

**Caveat**: The model is trained on modern English, so POS/dependency accuracy declines for older texts. Treat these as *approximations* of historical trends.


## Summary of Saved Outputs
 
All analysis results have been saved as CSV files in `analysis/tables/`:
 - `nb05-pos_distribution.csv`
 - `nb05-adj_noun_relations.csv`
 - `nb05-svo_triples.csv`
 - `nb05-noun_compounds.csv`
 - `nb05-cooccurrence_pairs.csv`
 
Figures are saved in `analysis/figures/`:
 - `nb05-adj_noun_temporal.png`
 - `nb05-svo_temporal.png`
 - `nb05-compounds_temporal.png`
 
These tables and plots can now be used for visualization, statistical testing, or as features
for further modeling (e.g., topic modeling, classification, or network analysis).

```mermaid
flowchart TB
    A0["00<br/>Bootcamp"] --> P1

    subgraph P1["Part I — Corpus building and analysis"]
        direction LR
        A1a["01a<br/>Corpus metadata"] --> A1b["01b<br/>Corpus building"] --> A2["02<br/>Preprocessing"] --> A3["03<br/>Distributions + time"] --> A4a["04a<br/>Lexical exploration"] --> A4b["04b<br/>Embedding"]
    end

    subgraph P2["Part II — Linguistic annotations"]
        direction LR
        A5a["05a<br/>spaCy annotation"] --> A5b["05b<br/>Relation extraction"] --> A6a["06a<br/>NER"] --> A6b["06b<br/>Custom NER"]
    end

    subgraph P3["Part III — Representations"]
        direction LR
        A7["07<br/>BoW + TF-IDF"] --> A8a["08a<br/>Embeddings"] --> A8b["08b<br/>Transformers"]
    end

    subgraph P4["Part IV — Models and interpretation"]
        direction LR
        A9["09<br/>Classification"] --> A10["10<br/>Custom NER training"] --> A11["11<br/>Topic modeling"] --> A12["12<br/>Semantic shift"]
    end

    P1 --> P2
    P2 --> P3
    P3 --> P4

    classDef start fill:#f3f0ff,stroke:#6f42c1,stroke-width:1.5px,color:#111;
    classDef prep fill:#eef7ff,stroke:#1f77b4,stroke-width:1.5px,color:#111;
    classDef annot fill:#eefaf0,stroke:#2ca02c,stroke-width:1.5px,color:#111;
    classDef repr fill:#fff7e6,stroke:#ff8c00,stroke-width:1.5px,color:#111;
    classDef model fill:#fff0f0,stroke:#d62728,stroke-width:1.5px,color:#111;

    classDef highlight fill:#fff3b0,stroke:#f5a623,stroke-width:4px,color:#111;

    class A1a,A1b,A2,A3,A4a,A4b prep;
    class A5a,A5b,A6a,A6b annot;
    class A7,A8a,A8b repr;
    class A9,A10,A11,A12 model;

    class A5b highlight;
```